# NB01_clean — Làm sạch & tiền xử lý dữ liệu POS (mục 3.4)
**Cách chạy:** mở trên Colab → Runtime → Run all. Không cần mount Drive hay tạo lối tắt.
- Đầu vào: tải thẳng `data/coffee_shop_sales_raw.csv` từ thư mục Drive công khai của nhóm (theo ID file).
- Đầu ra: `staging/sales_clean.csv`, `staging/products_drinks.csv`, `staging/bang_3_4.md`.
  Ô cuối hỏi đăng nhập Google một lần rồi đẩy 3 file vào thư mục `staging/` trên Drive của nhóm
  (ghi đè bản cũ cùng tên, không tạo bản trùng).
Nguyên tắc: không xóa dòng nào; ngoại lai chỉ gắn cờ.

In [2]:
# ==== 0. CẤU HÌNH ====
import os, hashlib
import numpy as np
import pandas as pd

# ID lấy từ link Drive: drive.google.com/drive/folders/<ID> và drive.google.com/file/d/<ID>
DRIVE_FOLDER_ID = '1Z6FJnMgtthexz27Rho5sdX5UBCIdtmYa'   # BI_K24406H_Kỳ7 (ai có link đều sửa được)
RAW_FILE_ID = '14t3nlw6cO1ju0qG7HXzWONE6XOaC9B9m'       # data/coffee_shop_sales_raw.csv
UPLOAD_TO_DRIVE = True                                   # False = chỉ lưu trong Colab

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK = '/content' if IN_COLAB else os.environ.get('NB01_BASE', '.')
RAW_PATH = os.path.join(WORK, 'data', 'coffee_shop_sales_raw.csv')
OUT_DIR = os.path.join(WORK, 'staging')
os.makedirs(os.path.dirname(RAW_PATH), exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(RAW_PATH):
    import gdown   # có sẵn trên Colab; máy khác: pip install gdown
    gdown.download(id=RAW_FILE_ID, output=RAW_PATH, quiet=False)

EXPECTED_SHA256 = '287d06d19f7a832d48f1deea7b149836fb32d591010f060bdcf8d1959653bfe1'
EXPECTED_ROWS = 149_116
EXPECTED_REVENUE = 698_812.33
EXPECTED_DRINK_REVENUE = 538_774.40     # Bảng 3.2.7
EXPECTED_DRINK_PRODUCTS = 41
DRINK_CATEGORIES = ['Coffee', 'Tea', 'Drinking Chocolate']
SIZE_SUFFIX = {'Sm': 'Sm', 'Rg': 'Rg', 'Lg': 'Lg'}
# Ngoại lệ tên không có hậu tố size (PCCV 3.4, bước 3)
SIZE_EXCEPTIONS = {
    'Cappuccino':           ('Rg',   3.75),
    'Latte':                ('Sm',   3.75),
    'Espresso shot':        ('Shot', None),
    'Ouro Brasileiro shot': ('Shot', None),
}
SHIFTS = [('Sáng', 6, 10), ('Chiều', 11, 15), ('Tối', 16, 20)]
print('RAW_PATH =', RAW_PATH)

RAW_PATH = /content/data/coffee_shop_sales_raw.csv


In [3]:
# ==== 1. ĐỌC DỮ LIỆU GỐC & KIỂM TRA TOÀN VẸN ====
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

file_hash = sha256_of(RAW_PATH)
if RAW_PATH.endswith('.xlsx'):
    hash_ok = file_hash == EXPECTED_SHA256
    print('SHA-256:', file_hash, '| khớp 3.3.2:', hash_ok)
    raw = pd.read_excel(RAW_PATH, dtype={'transaction_time': str})
else:
    hash_ok = None   # CSV xuất lại từ xlsx: hash khác bản gốc; toàn vẹn kiểm bằng số dòng + tổng doanh thu bên dưới
    print('Nguồn CSV:', RAW_PATH, '| SHA-256 của CSV:', file_hash)
    raw = pd.read_csv(RAW_PATH, dtype={'transaction_time': str}, keep_default_na=False, na_values=[''])
print(raw.shape)
print(raw.dtypes)
assert len(raw) == EXPECTED_ROWS, f'Số dòng gốc {len(raw)} != {EXPECTED_ROWS}'

issues = []   # (vấn đề, cách xử lý, số dòng ảnh hưởng) -> Bảng 3.x
df = raw.copy()

Nguồn CSV: /content/data/coffee_shop_sales_raw.csv | SHA-256 của CSV: 4edb90a59db117477d9be5a40d9a600bc9cb872d08609119f99eb94da4bbf6be
(149116, 11)
transaction_id        int64
transaction_date     object
transaction_time     object
transaction_qty       int64
store_id              int64
store_location       object
product_id            int64
unit_price          float64
product_category     object
product_type         object
product_detail       object
dtype: object


In [4]:
# ==== 3.4.1. CHUẨN HÓA ĐỊNH DẠNG & KIỂU DỮ LIỆU ====
# (a) TRIM mọi cột chữ
text_cols = [c for c in df.columns if df[c].dtype == object or pd.api.types.is_string_dtype(df[c])]
text_cols = [c for c in text_cols if c != 'transaction_time']
trim_detail = {}
rows_trimmed = pd.Series(False, index=df.index)
for c in text_cols:
    s = df[c].astype('string')
    stripped = s.str.strip().str.replace(r'\s+', ' ', regex=True)
    changed = (s != stripped).fillna(False)
    if changed.any():
        trim_detail[c] = {
            'so_dong': int(changed.sum()),
            'gia_tri': sorted(s[changed].unique().tolist()),
        }
    rows_trimmed |= changed
    df[c] = stripped
print('Cột có khoảng trắng thừa:', trim_detail)
for c, d in trim_detail.items():
    vals = ', '.join(f'"{v}"' for v in d['gia_tri'])
    issues.append((f'Khoảng trắng thừa ở cột {c} ({vals})',
                   'TRIM và gộp khoảng trắng kép trên mọi cột chữ', d['so_dong']))

# (b) transaction_time: văn bản -> kiểu thời gian
t = pd.to_datetime(df['transaction_time'].astype(str).str.strip(), format='%H:%M:%S', errors='coerce')
bad_time = int(t.isna().sum())
assert bad_time == 0, f'{bad_time} dòng giờ không đọc được'
df['transaction_time'] = t.dt.time
df['transaction_date'] = pd.to_datetime(df['transaction_date']).dt.normalize()
df['transaction_datetime'] = df['transaction_date'] + pd.to_timedelta(t.dt.strftime('%H:%M:%S'))
issues.append(('transaction_time lưu dạng văn bản HH:MM:SS',
               'Chuyển sang kiểu thời gian; ghép thêm transaction_datetime', len(df)))

# (c) Kiểu số & kiểm tra miền giá trị
df['transaction_id'] = df['transaction_id'].astype('int64')
df['transaction_qty'] = df['transaction_qty'].astype('int64')
df['store_id'] = df['store_id'].astype('int64')
df['product_id'] = df['product_id'].astype('int64')
df['unit_price'] = df['unit_price'].astype('float64')

checks_141 = {
    'gia_tri_thieu': int(df[raw.columns].isna().sum().sum()),
    'dong_trung_lap': int(df.duplicated(subset=[c for c in raw.columns if c != 'transaction_id']).sum()),
    'transaction_id_trung': int(df['transaction_id'].duplicated().sum()),
    'qty_<=0': int((df['transaction_qty'] <= 0).sum()),
    'gia_<=0': int((df['unit_price'] <= 0).sum()),
    'ngay_min': df['transaction_date'].min().date().isoformat(),
    'ngay_max': df['transaction_date'].max().date().isoformat(),
    'gio_min': str(min(df['transaction_time'])),
    'gio_max': str(max(df['transaction_time'])),
}
print(checks_141)

# 1 product_id ↔ 1 tên? (sau TRIM)
name_per_id = df.groupby('product_id')['product_detail'].nunique()
assert (name_per_id == 1).all(), name_per_id[name_per_id > 1]

# Sản phẩm nhiều mức giá -> giữ đơn giá ở mức dòng
price_per_id = df.groupby('product_id')['unit_price'].nunique()
multi_price_ids = price_per_id[price_per_id > 1].index
rows_multi_price = int(df['product_id'].isin(multi_price_ids).sum())
issues.append((f'{len(multi_price_ids)} sản phẩm có nhiều hơn một mức giá',
               'Giữ unit_price ở mức dòng giao dịch; giá danh mục = bình quân theo số lượng',
               rows_multi_price))

Cột có khoảng trắng thừa: {'product_detail': {'so_dong': 1952, 'gia_tri': ['Scottish Cream Scone ']}}
{'gia_tri_thieu': 0, 'dong_trung_lap': 398, 'transaction_id_trung': 0, 'qty_<=0': 0, 'gia_<=0': 0, 'ngay_min': '2023-01-01', 'ngay_max': '2023-06-30', 'gio_min': '06:00:00', 'gio_max': '20:59:32'}


In [5]:
# ==== 3.4.2. TRƯỜNG DẪN XUẤT ====
dt = df['transaction_datetime']
df['hour'] = dt.dt.hour.astype('int64')
df['day_of_week'] = dt.dt.dayofweek.add(1).astype('int64')        # 1 = Thứ Hai ... 7 = Chủ nhật (ISO)
df['day_name'] = dt.dt.day_name()
df['week_start'] = (df['transaction_date'] - pd.to_timedelta(dt.dt.dayofweek, unit='D')).dt.date
df['month'] = dt.dt.month.astype('int64')

def to_shift(h):
    for name, lo, hi in SHIFTS:
        if lo <= h <= hi:
            return name
    return pd.NA
df['shift'] = df['hour'].map(to_shift)
no_shift = int(df['shift'].isna().sum())
assert no_shift == 0, f'{no_shift} dòng ngoài 06–20h'

df['revenue'] = (df['transaction_qty'] * df['unit_price']).round(2)

week_counts = df.groupby('week_start')['transaction_date'].nunique()
partial_weeks = week_counts[week_counts < 7]
print('Số tuần:', len(week_counts), '| tuần biên:', partial_weeks.to_dict())

Số tuần: 27 | tuần biên: {datetime.date(2022, 12, 26): 1, datetime.date(2023, 6, 26): 5}


In [6]:
# ---- Tách size / base_item từ product_detail ----
def split_size(detail, price):
    parts = detail.rsplit(' ', 1)
    if len(parts) == 2 and parts[1] in SIZE_SUFFIX:
        return parts[0], SIZE_SUFFIX[parts[1]], 'hau_to'
    if detail in SIZE_EXCEPTIONS:
        size, _ = SIZE_EXCEPTIONS[detail]
        return detail, size, 'ngoai_le'
    return detail, pd.NA, 'khong_size'

df['is_drink'] = df['product_category'].isin(DRINK_CATEGORIES)
prod = (df.groupby('product_id')
          .agg(product_detail=('product_detail', 'first'),
               product_category=('product_category', 'first'),
               product_type=('product_type', 'first'),
               n_prices=('unit_price', 'nunique'),
               price_min=('unit_price', 'min'),
               price_max=('unit_price', 'max'))
          .reset_index())
prod[['base_item', 'size', 'size_rule']] = prod.apply(
    lambda r: pd.Series(split_size(r['product_detail'], r['price_min'])), axis=1)
prod['is_drink'] = prod['product_category'].isin(DRINK_CATEGORIES)

# Kiểm ngoại lệ có đúng mức giá ghi trong PCCV
for name, (size, price) in SIZE_EXCEPTIONS.items():
    row = prod[prod['product_detail'] == name]
    assert len(row) == 1, f'Không thấy sản phẩm ngoại lệ "{name}"'
    if price is not None:
        assert np.isclose(row['price_min'].iloc[0], price) and np.isclose(row['price_max'].iloc[0], price), \
            f'"{name}" có giá {row[["price_min","price_max"]].values} khác {price}'

# Đồ uống thì bắt buộc có size; hàng khác để NA
drink_no_size = prod[prod['is_drink'] & prod['size'].isna()]
assert drink_no_size.empty, drink_no_size[['product_id', 'product_detail']]
non_drink_with_size = prod[~prod['is_drink'] & prod['size'].notna()]
print('Hàng không phải đồ uống nhưng có hậu tố size:\n', non_drink_with_size[['product_detail', 'product_category']])

df = df.merge(prod[['product_id', 'base_item', 'size', 'size_rule']], on='product_id', how='left',
              validate='many_to_one')
assert len(df) == EXPECTED_ROWS

size_rows = df[df['is_drink']].groupby('size_rule').size().to_dict()
exc_rows = df[df['size_rule'] == 'ngoai_le'].groupby('product_detail').size().to_dict()
issues.append(('Kích cỡ nằm chung trong tên sản phẩm (hậu tố Sm/Rg/Lg)',
               'Tách hậu tố → size, phần còn lại → base_item', size_rows.get('hau_to', 0)))
issues.append(('Đồ uống không ghi kích cỡ: ' + ', '.join(f'{k} ({v})' for k, v in exc_rows.items()),
               'Gán theo quy ước: Cappuccino 3,75$ = Rg; Latte 3,75$ = Sm; hai loại shot = Shot',
               size_rows.get('ngoai_le', 0)))
issues.append(('Hàng không pha chế không có kích cỡ',
               'Để size = NA, base_item = product_detail', int((~df['is_drink']).sum())))
print(prod[prod['is_drink']].groupby('size').size())

Hàng không phải đồ uống nhưng có hậu tố size:
 Empty DataFrame
Columns: [product_detail, product_category]
Index: []
size
Lg      16
Rg      17
Shot     2
Sm       6
dtype: int64


In [7]:
# ==== 3.4.3. NGOẠI LAI THEO IQR (chỉ gắn cờ) ====
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1, q3, iqr, q1 - 1.5 * iqr, q3 + 1.5 * iqr

q1, q3, iqr, lo, hi = iqr_bounds(df['revenue'])
df['outlier_flag'] = ((df['revenue'] < lo) | (df['revenue'] > hi)).astype('int8')
n_out = int(df['outlier_flag'].sum())
print(f'Q1={q1:.2f} Q3={q3:.2f} IQR={iqr:.2f} | ngưỡng [{lo:.2f}; {hi:.2f}] | cờ: {n_out} dòng')

out_by_cat = (df.groupby('product_category')
                .agg(so_dong=('outlier_flag', 'size'), dong_ngoai_lai=('outlier_flag', 'sum'),
                     doanh_thu_ngoai_lai=('revenue', lambda s: s[df.loc[s.index, 'outlier_flag'] == 1].sum()))
                .sort_values('dong_ngoai_lai', ascending=False))
out_by_cat['ty_le_%'] = (out_by_cat['dong_ngoai_lai'] / out_by_cat['so_dong'] * 100).round(2)
print(out_by_cat)

# Đối chiếu: IQR riêng trong tập đồ uống (không ghi thành cột, chỉ báo cáo)
d_q1, d_q3, d_iqr, d_lo, d_hi = iqr_bounds(df.loc[df['is_drink'], 'revenue'])
n_out_drink_global = int(df.loc[df['is_drink'], 'outlier_flag'].sum())
n_out_drink_local = int(((df['revenue'] > d_hi) | (df['revenue'] < d_lo))[df['is_drink']].sum())
print(f'Đồ uống: cờ theo ngưỡng toàn bộ = {n_out_drink_global}; theo IQR riêng [{d_lo:.2f}; {d_hi:.2f}] = {n_out_drink_local}')
print('Phân bố qty của dòng bị cờ:', df.loc[df['outlier_flag'] == 1, 'transaction_qty'].value_counts().sort_index().to_dict())
issues.append((f'Doanh thu dòng nằm ngoài [Q1 − 1,5·IQR; Q3 + 1,5·IQR] = [{lo:.2f}; {hi:.2f}] USD',
               'Gắn outlier_flag = 1, giữ nguyên dòng (giao dịch hợp lệ: hàng giá cao hoặc mua nhiều)', n_out))

Q1=3.00 Q3=6.00 IQR=3.00 | ngưỡng [-1.50; 10.50] | cờ: 3273 dòng
                    so_dong  dong_ngoai_lai  doanh_thu_ngoai_lai  ty_le_%
product_category                                                         
Coffee beans           1753            1619             38745.25    92.36
Branded                 747             747             13607.00   100.00
Coffee                58416             378              4489.00     0.65
Drinking Chocolate    11468             182              2364.00     1.59
Packaged Chocolate      487             148              1972.84    30.39
Loose Tea              1210             122              1335.90    10.08
Tea                   45449              77               924.00     0.17
Bakery                22796               0                 0.00     0.00
Flavours               6790               0                 0.00     0.00
Đồ uống: cờ theo ngưỡng toàn bộ = 637; theo IQR riêng [-1.50; 10.50] = 637
Phân bố qty của dòng bị cờ: {1: 2608, 2: 13, 3

In [8]:
# ==== 3.4.4. TẬP 41 ĐỒ UỐNG PHA CHẾ ====
drinks = prod[prod['is_drink']].copy()
assert len(drinks) == EXPECTED_DRINK_PRODUCTS, len(drinks)

agg = (df[df['is_drink']].groupby('product_id')
         .agg(qty_sold=('transaction_qty', 'sum'), revenue=('revenue', 'sum'), n_lines=('transaction_id', 'size'))
         .reset_index())
drinks = drinks.merge(agg, on='product_id', how='left', validate='one_to_one')
drinks['avg_unit_price'] = drinks['revenue'] / drinks['qty_sold']     # giá bình quân theo số lượng (3.2.2)
drinks['menu_mix_pct'] = drinks['qty_sold'] / drinks['qty_sold'].sum() * 100
products_drinks = drinks[['product_id', 'product_category', 'product_type', 'product_detail', 'base_item', 'size',
                          'n_prices', 'price_min', 'price_max', 'avg_unit_price',
                          'qty_sold', 'n_lines', 'revenue', 'menu_mix_pct']].sort_values('product_id')
print(products_drinks.groupby('product_category').size())
print('Số base_item khác nhau:', products_drinks['base_item'].nunique())
print('MM% nhỏ nhất:', round(products_drinks['menu_mix_pct'].min(), 6))

product_category
Coffee                21
Drinking Chocolate     4
Tea                   16
dtype: int64
Số base_item khác nhau: 19
MM% nhỏ nhất: 1.848745


In [9]:
# ==== 3.4.5. KIỂM TRA & GHI FILE ====
total_rev = round(df['revenue'].sum(), 2)
drink_rev = round(df.loc[df['is_drink'], 'revenue'].sum(), 2)
check = pd.DataFrame([
    ('Số dòng', len(df), EXPECTED_ROWS),
    ('Tổng revenue (USD)', total_rev, EXPECTED_REVENUE),
    ('Revenue 41 đồ uống (USD)', drink_rev, EXPECTED_DRINK_REVENUE),
    ('Số sản phẩm đồ uống', len(products_drinks), EXPECTED_DRINK_PRODUCTS),
    ('Dòng thiếu shift', no_shift, 0),
    ('Đồ uống thiếu size', len(drink_no_size), 0),
    ('Số ngày có giao dịch', df['transaction_date'].nunique(), 181),
], columns=['Phép kiểm', 'Kết quả', 'Kỳ vọng'])
check['Đạt'] = [np.isclose(float(a), float(b)) for a, b in zip(check['Kết quả'], check['Kỳ vọng'])]
print(check.to_string(index=False))
assert check['Đạt'].all(), 'Có phép kiểm không đạt — không ghi file'

SALES_COLS = ['transaction_id', 'transaction_date', 'transaction_time', 'transaction_datetime',
              'store_id', 'store_location', 'product_id', 'product_category', 'product_type', 'product_detail',
              'base_item', 'size', 'transaction_qty', 'unit_price', 'revenue',
              'hour', 'day_of_week', 'day_name', 'week_start', 'month', 'shift',
              'is_drink', 'outlier_flag']
out = df[SALES_COLS].copy()
out['transaction_date'] = out['transaction_date'].dt.strftime('%Y-%m-%d')
out['transaction_time'] = out['transaction_time'].astype(str)
out['transaction_datetime'] = out['transaction_datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
out['is_drink'] = out['is_drink'].astype('int8')
out.to_csv(os.path.join(OUT_DIR, 'sales_clean.csv'), index=False, encoding='utf-8')
products_drinks.round({'avg_unit_price': 6, 'revenue': 2, 'menu_mix_pct': 6}) \
    .to_csv(os.path.join(OUT_DIR, 'products_drinks.csv'), index=False, encoding='utf-8')

# Đọc lại để chắc file ghi đúng
chk = pd.read_csv(os.path.join(OUT_DIR, 'sales_clean.csv'))
assert len(chk) == EXPECTED_ROWS and np.isclose(round(chk['revenue'].sum(), 2), EXPECTED_REVENUE)
print('Đã ghi', OUT_DIR)

               Phép kiểm   Kết quả   Kỳ vọng  Đạt
                 Số dòng 149116.00 149116.00 True
      Tổng revenue (USD) 698812.33 698812.33 True
Revenue 41 đồ uống (USD) 538774.40 538774.40 True
     Số sản phẩm đồ uống     41.00     41.00 True
        Dòng thiếu shift      0.00      0.00 True
      Đồ uống thiếu size      0.00      0.00 True
    Số ngày có giao dịch    181.00    181.00 True
Đã ghi /content/staging


In [10]:
# ==== BẢNG CHO BÁO CÁO (dán vào mục 3.4.5) ====
fmt = lambda x: f'{x:,}'.replace(',', '.') if isinstance(x, (int, np.integer)) else x
lines = ['| STT | Vấn đề | Cách xử lý | Số dòng ảnh hưởng |', '|:-:|---|---|--:|']
for i, (a, b, n) in enumerate(issues, 1):
    lines.append(f'| {i} | {a} | {b} | {fmt(int(n))} |')
lines += ['', '| Phép kiểm | Kết quả | Kỳ vọng | Đạt |', '|---|--:|--:|:-:|']
for _, r in check.iterrows():
    num = lambda v: f'{int(v):,}'.replace(',', '.') if float(v).is_integer() else f'{v:,.2f}'.replace(',', ' ').replace('.', ',').replace(' ', '.')
    lines.append(f"| {r['Phép kiểm']} | {num(r['Kết quả'])} | {num(r['Kỳ vọng'])} | {'✔' if r['Đạt'] else '✘'} |")
lines += ['', f'Nguồn: {os.path.basename(RAW_PATH)} | SHA-256 khớp 3.3.2: {hash_ok if hash_ok is not None else "không áp dụng (CSV)"}',
          f'Tuần biên: {dict((str(k), int(v)) for k, v in partial_weeks.items())}',
          f'IQR toàn bộ: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, ngưỡng trên={hi:.2f}',
          f'IQR riêng đồ uống: ngưỡng trên={d_hi:.2f}; đồ uống bị cờ theo ngưỡng toàn bộ={n_out_drink_global}, theo IQR riêng={n_out_drink_local}',
          '', 'Ngoại lai theo nhóm sản phẩm:', out_by_cat.to_markdown()]
report = '\n'.join(lines)
open(os.path.join(OUT_DIR, 'bang_3_4.md'), 'w', encoding='utf-8').write(report)
print(report)

| STT | Vấn đề | Cách xử lý | Số dòng ảnh hưởng |
|:-:|---|---|--:|
| 1 | Khoảng trắng thừa ở cột product_detail ("Scottish Cream Scone ") | TRIM và gộp khoảng trắng kép trên mọi cột chữ | 1.952 |
| 2 | transaction_time lưu dạng văn bản HH:MM:SS | Chuyển sang kiểu thời gian; ghép thêm transaction_datetime | 149.116 |
| 3 | 15 sản phẩm có nhiều hơn một mức giá | Giữ unit_price ở mức dòng giao dịch; giá danh mục = bình quân theo số lượng | 24.159 |
| 4 | Kích cỡ nằm chung trong tên sản phẩm (hậu tố Sm/Rg/Lg) | Tách hậu tố → size, phần còn lại → base_item | 104.598 |
| 5 | Đồ uống không ghi kích cỡ: Cappuccino (2793), Espresso shot (2730), Latte (2990), Ouro Brasileiro shot (2222) | Gán theo quy ước: Cappuccino 3,75$ = Rg; Latte 3,75$ = Sm; hai loại shot = Shot | 10.735 |
| 6 | Hàng không pha chế không có kích cỡ | Để size = NA, base_item = product_detail | 33.783 |
| 7 | Doanh thu dòng nằm ngoài [Q1 − 1,5·IQR; Q3 + 1,5·IQR] = [-1.50; 10.50] USD | Gắn outlier_flag = 1, giữ nguyên dòng (gi

In [11]:
# ==== ĐẨY KẾT QUẢ LÊN DRIVE CỦA NHÓM (staging/) ====
def upload_to_drive(paths, parent_id, subfolder='staging'):
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    auth.authenticate_user()            # đăng nhập Google một lần; tài khoản nào cũng được vì thư mục mở quyền sửa
    svc = build('drive', 'v3', cache_discovery=False)

    def find(name, parent, folder=False):
        q = f"name = '{name}' and '{parent}' in parents and trashed = false"
        if folder:
            q += " and mimeType = 'application/vnd.google-apps.folder'"
        r = svc.files().list(q=q, fields='files(id, name)', supportsAllDrives=True,
                             includeItemsFromAllDrives=True).execute()
        return r['files'][0]['id'] if r['files'] else None

    folder_id = find(subfolder, parent_id, folder=True)
    if folder_id is None:
        folder_id = svc.files().create(body={'name': subfolder, 'parents': [parent_id],
                                             'mimeType': 'application/vnd.google-apps.folder'},
                                       fields='id', supportsAllDrives=True).execute()['id']
    for p in paths:
        name = os.path.basename(p)
        media = MediaFileUpload(p, mimetype='text/csv' if name.endswith('.csv') else 'text/markdown', resumable=True)
        fid = find(name, folder_id)
        if fid:   # ghi đè bản cũ, giữ nguyên link
            svc.files().update(fileId=fid, media_body=media, supportsAllDrives=True).execute()
            print('Cập nhật:', name)
        else:
            fid = svc.files().create(body={'name': name, 'parents': [folder_id]}, media_body=media,
                                     fields='id', supportsAllDrives=True).execute()['id']
            print('Tạo mới:', name)
        print('   https://drive.google.com/file/d/' + fid)
    print('Thư mục staging: https://drive.google.com/drive/folders/' + folder_id)

outputs = [os.path.join(OUT_DIR, f) for f in ('sales_clean.csv', 'products_drinks.csv', 'bang_3_4.md')]
if IN_COLAB and UPLOAD_TO_DRIVE:
    upload_to_drive(outputs, DRIVE_FOLDER_ID)
else:
    print('Không đẩy lên Drive; file nằm ở', OUT_DIR)

Tạo mới: sales_clean.csv
   https://drive.google.com/file/d/1XZYQ7rg0pblDnwYuR505OHlrfd0lXSBH
Tạo mới: products_drinks.csv
   https://drive.google.com/file/d/1RLQdVnBkWOdeXrDLejEVwztAhY8JQcGf
Tạo mới: bang_3_4.md
   https://drive.google.com/file/d/1elGnpE2CNbMBRgQor3wk2pyp5WolJDDQ
Thư mục staging: https://drive.google.com/drive/folders/13fw9VsoTPgKDMuthRFQccCqQXX0NabVu
